# PC-TTP Alignment Hypothesis Validation

Cheap validation pass for `20260813-pc_ttp_anchored_curve_resampler_v2.md`, **before**
touching `04_cross_dataset_training.py`/`combine_group` for real: does aligning chips
by their PC well's time-to-positivity (instead of raw acquisition-start zeroing) make
the *same target* actually converge across chips? If PC itself doesn't converge after
alignment, timestamp misalignment isn't (or isn't the whole) story.

This notebook does **not** modify any existing `.py` file — it imports
`04_cross_dataset_training.py` (`combine_group`'s sibling helpers, `CurveResampler`)
and `config.py` as-is, and implements the new PC-TTP alignment logic entirely here,
matching `combine_group`'s exact output shape so the existing `plot_target_across_chips`
(`lofo_class_balance_analysis.ipynb` §10) can be reused unchanged on the result.

**Decisions already made** (per discussion, not re-litigated here):
- Anchor is **fold-scoped**: computed from every chip *except* whichever one is being
  treated as "held out" for that comparison, then applied (with graceful clipping) to
  every chip including the held-out one — this is also how we test the "unseen chip
  has an earlier TTP than the anchor" edge case for real, not hypothetically.
- Both `anchor_method="min"` and `anchor_method="percentile"` are implemented as a
  parameter, to compare visually rather than assume one is better.
- This is a **validation-only** notebook. Whether this becomes a real opt-in mode in
  `04_cross_dataset_training.py` is a separate decision, made after inspecting these
  results — nothing here is wired into the production pipeline.

In [2]:
import sys
import importlib
import colorsys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [3]:
import os

try:
    # VS Code injects '__vsc_ipynb_file__' into the globals automatically
    notebook_path = globals().get('__vsc_ipynb_file__')
    if notebook_path:
        notebook_dir = os.path.dirname(os.path.dirname(notebook_path))
        os.chdir(notebook_dir)
except Exception as e:
    print(f"Could not change directory: {e}")

print("Current Working Directory:", os.getcwd())
%load_ext autoreload
%autoreload 2
import config

# Import-only, per instructions -- 04_cross_dataset_training.py itself is never
# modified. cdt's own imports already put utils/model_training on sys.path.
cdt = importlib.import_module("04_cross_dataset_training")
from model_utils import CurveResampler   # same resampler class combine_group uses

%matplotlib inline


Current Working Directory: /vol/bitbucket/gk225/POC_DDM/gk_code/main

[*] SUCCESS: TensorFlow is utilizing the GPU -> /physical_device:GPU:0



## 1. Configuration

In [4]:
GROUP_NAME = "final_4_chip_clean"
EXP_FOLDER = config.FINAL_EXP_FOLDER + "_nc_subtract"
CURVE_TYPE = "ori_curve_wavelet_bior35_norm"   # raw curves -- matches lofo_class_balance_analysis.ipynb's default

folder_names = config.CROSS_DATASET_GROUPS[GROUP_NAME]
exp_paths = [Path(EXP_FOLDER, name) for name in folder_names]

def short_name(folder):
    return folder.split('_U_', 1)[1]

print(f"Group '{GROUP_NAME}' -> {len(folder_names)} chips:")
for n in folder_names:
    print(" -", n, f"({short_name(n)})")


Group 'final_4_chip_clean' -> 4 chips:
 - D20260806_E00_C00_F4500KHz_U_DDM_01_06 (DDM_01_06)
 - D20260807_E00_C00_F4500KHz_U_DDM_02_07 (DDM_02_07)
 - D20260808_E00_C00_F4500KHz_U_DDM_03_01 (DDM_03_01)
 - D20260810_E00_C00_F4500KHz_U_DDM_04_01 (DDM_04_01)


## 2. Load PC ground truth + compute TTP per chip

`load_curve_data()` applies `config.apply_well_exclusion`, which strips PC rows
(excluded from the classifier because "no point predicting positive control" — not a
data-quality exclusion). `load_full_curve_data` below bypasses that specifically to
get PC back, using the same temporarily-clear-and-restore trick already used in
`lofo_class_balance_analysis.ipynb`'s `label_counts_for_exp(..., apply_exclusion=False)`.

TTP = `Ct` from `features_df` — the sigmoid-fit 20%-of-range crossing time, already
computed by `02_outlier_detection_pipeline.py` via `sp.extract_kinetic_parameters_original`
(the same underlying function `multi_ttp_per_variant_analysis.ipynb`'s
`get_ttp(method="ct_idx")` resolves to) — no new fitting code needed.

In [5]:
def load_full_curve_data(exp_path, curve_type):
    """cdt.load_curve_data with EXCLUDE_WELL_MAPPING bypassed, so PC rows survive."""
    saved = dict(config.EXCLUDE_WELL_MAPPING)
    config.EXCLUDE_WELL_MAPPING.clear()
    try:
        return cdt.load_curve_data(exp_path, curve_type)
    finally:
        config.EXCLUDE_WELL_MAPPING.clear()
        config.EXCLUDE_WELL_MAPPING.update(saved)


def pc_ttp_per_chip(exp_paths, curve_type, ct_col="Ct"):
    """{chip_name: mean Ct over that chip's PC-labeled rows}."""
    ttp = {}
    for exp_path in exp_paths:
        d = load_full_curve_data(exp_path, curve_type)
        if d is None:
            print(f"  [!] {exp_path.name}: could not load.")
            continue
        pc_mask = d["Y_mapped"] == "PC"
        if not pc_mask.any():
            print(f"  [!] {exp_path.name}: no PC-labeled rows found.")
            continue
        if ct_col not in d["features_df"].columns:
            print(f"  [!] {exp_path.name}: '{ct_col}' not in features_df.")
            continue
        ttp[exp_path.name] = float(d["features_df"][ct_col].values[pc_mask].mean())
    return ttp


pc_ttp = pc_ttp_per_chip(exp_paths, CURVE_TYPE)
print("\nPC TTP (mean Ct) per chip:")
for name, v in sorted(pc_ttp.items(), key=lambda kv: kv[1]):
    print(f"  {short_name(name):>12s}: {v:.2f}")
print(f"\nSpread: {max(pc_ttp.values()) - min(pc_ttp.values()):.2f} "
      f"(min={min(pc_ttp.values()):.2f}, max={max(pc_ttp.values()):.2f})")
print("If this spread is tiny relative to the curve duration, timestamp misalignment "
      "probably isn't a meaningful factor here -- worth checking before reading too "
      "much into the figures below.")


  [!] D20260806_E00_C00_F4500KHz_U_DDM_01_06: no PC-labeled rows found.
  [!] D20260807_E00_C00_F4500KHz_U_DDM_02_07: no PC-labeled rows found.
  [!] D20260808_E00_C00_F4500KHz_U_DDM_03_01: no PC-labeled rows found.
  [!] D20260810_E00_C00_F4500KHz_U_DDM_04_01: no PC-labeled rows found.

PC TTP (mean Ct) per chip:


ValueError: max() iterable argument is empty

## 3. PC-TTP-aligned `combine_group` (fold-scoped, parameterized anchor)

Mirrors `cdt.combine_group` structurally (same output keys/shapes, so
`plot_target_across_chips` works on it unchanged), with one inserted step before the
existing `CurveResampler.fit`/`.transform()` call: each chip's curves are truncated at
the start so its own PC TTP lands at the fold-scoped anchor (clipped, never negative —
see the design doc §3), then truncated at the end to a fold-scoped common duration.
`CurveResampler` itself is untouched, just fed pre-aligned input instead of
raw-acquisition-zeroed input.

In [ ]:
def combine_group_pc_aligned(exp_paths, curve_type, held_out_chip=None,
                             anchor_method="min", anchor_pct=10, verbose=True):
    """Like cdt.combine_group, but PC-inclusive and PC-TTP-aligned before resampling.

    held_out_chip: chip name excluded from anchor/common-duration computation (fold
    -scoped leakage avoidance -- see design doc §4), but still aligned+included in the
    output using the anchor computed from the OTHER chips, so its behaviour as a
    genuinely unseen chip can be inspected directly.
    anchor_method: "min" (strict earliest PC TTP) or "percentile" (anchor_pct-th
    percentile of PC TTP) across the non-held-out chips.

    Returns (combined, pc_ttp, shifts, anchor) -- combined has the same keys as
    cdt.combine_group's return value.
    """
    parts = []
    for exp_path in exp_paths:
        d = load_full_curve_data(exp_path, curve_type)
        if d is not None:
            parts.append(d)
    if len(parts) < 2:
        print("  -> fewer than 2 usable chips.")
        return None

    pc_ttp_local = {}
    for p in parts:
        pc_mask = p["Y_mapped"] == "PC"
        if pc_mask.any() and "Ct" in p["features_df"].columns:
            pc_ttp_local[p["dataset_id"]] = float(p["features_df"]["Ct"].values[pc_mask].mean())

    train_ttps = [v for k, v in pc_ttp_local.items() if k != held_out_chip]
    if not train_ttps:
        raise ValueError("No PC TTP available for any training chip.")
    if anchor_method == "min":
        anchor = min(train_ttps)
    elif anchor_method == "percentile":
        anchor = float(np.percentile(train_ttps, anchor_pct))
    else:
        raise ValueError(f"Unknown anchor_method: {anchor_method!r}")

    # --- front truncation: align each chip's PC TTP to the anchor (clip, never fail) ---
    aligned = []
    shifts = {}
    for p in parts:
        ttp = pc_ttp_local.get(p["dataset_id"])
        shift = max(ttp - anchor, 0.0) if ttp is not None else 0.0
        shifts[p["dataset_id"]] = shift
        start_idx = int(np.searchsorted(p["timestamps"], p["timestamps"][0] + shift))
        start_idx = min(start_idx, len(p["timestamps"]) - 1)
        p2 = dict(p)
        p2["timestamps"] = p["timestamps"][start_idx:]
        p2["curves"] = p["curves"][:, start_idx:]
        aligned.append(p2)

    # --- back truncation: common duration from training chips only, clip for others ---
    train_lens = [p["timestamps"][-1] - p["timestamps"][0]
                  for p in aligned if p["dataset_id"] != held_out_chip]
    common_duration = min(train_lens)
    for p2 in aligned:
        end_time = p2["timestamps"][0] + common_duration
        end_idx = min(int(np.searchsorted(p2["timestamps"], end_time)) + 1, len(p2["timestamps"]))
        p2["timestamps"] = p2["timestamps"][:end_idx]
        p2["curves"] = p2["curves"][:, :end_idx]

    if verbose:
        print(f"  anchor ({anchor_method}) = {anchor:.2f}  |  held_out = {short_name(held_out_chip) if held_out_chip else None}")
        for name, s in shifts.items():
            flag = " <- CLIPPED (TTP below anchor)" if pc_ttp_local.get(name, anchor) < anchor else ""
            print(f"    {short_name(name):>12s}: shift={s:8.2f}{flag}")
        print(f"  common_duration = {common_duration:.2f}")

    # --- resample onto one common grid: SAME CurveResampler cdt.combine_group uses ---
    timestamps_zeroed = [p["timestamps"] - p["timestamps"][0] for p in aligned]
    resampler = CurveResampler.fit(timestamps_zeroed)
    for p in aligned:
        p["curves"] = resampler.transform(p["timestamps"], p["curves"])

    if all(p["coords"] is not None for p in aligned):
        coords_combined = np.concatenate([p["coords"] for p in aligned], axis=0)
        well_ids_combined = np.concatenate([p["well_ids"] for p in aligned], axis=0)
    else:
        coords_combined, well_ids_combined = None, None

    conc_parts = []
    for p in aligned:
        raw = p.get("concentration_raw")
        conc_parts.append(np.asarray(raw, dtype=object) if raw is not None
                          else np.full(len(p["Y_mapped"]), None, dtype=object))

    combined = {
        "curves": np.concatenate([p["curves"] for p in aligned], axis=0),
        "features_df": pd.concat([p["features_df"] for p in aligned], axis=0, ignore_index=True),
        "Y_mapped": np.concatenate([p["Y_mapped"] for p in aligned], axis=0),
        "dataset_id": np.concatenate([np.full(len(p["Y_mapped"]), p["dataset_id"], dtype=object) for p in aligned], axis=0),
        "dataset_names": [p["dataset_id"] for p in aligned],
        "resampler": resampler,
        "coords": coords_combined,
        "well_ids": well_ids_combined,
        "concentration_raw": np.concatenate(conc_parts, axis=0),
    }
    return combined, pc_ttp_local, shifts, anchor


print("combine_group_pc_aligned defined.")


## 4. Visual check 1 — does the same target converge across chips after alignment?

`plot_target_across_chips`, `build_well_table`, `_shade`/`_exp_colors`/`_exp_legend`/
`_raw_conc` copied verbatim from `lofo_class_balance_analysis.ipynb` §5/§10 (not
importable across notebooks, so reused by copy per that notebook's own established
pattern for this). PC is included here (unlike in that notebook) since exclusion was
bypassed in §2/§3 above.

In [ ]:
def build_well_table(combined):
    """One row per globally-unique well, with its dataset, class label, and the
    full set of pixel row-indices belonging to it. Copied from
    lofo_class_balance_analysis.ipynb §5."""
    df = pd.DataFrame({
        "well_id": combined["well_ids"],
        "dataset_id": combined["dataset_id"],
        "label": combined["Y_mapped"],
    })
    df["row_idx"] = np.arange(len(df))
    meta = df.groupby("well_id").agg(dataset_id=("dataset_id", "first"), label=("label", "first"))
    row_idx_map = df.groupby("well_id")["row_idx"].apply(np.array)
    wells = meta.join(row_idx_map.rename("row_idx")).reset_index()
    wells["n_pixels"] = wells["row_idx"].apply(len)
    return wells


def _shade(base_rgb, frac, spread=0.3):
    """Copied verbatim from multi_ttp_per_variant_analysis.ipynb's
    plot_label_conc_grid, via lofo_class_balance_analysis.ipynb §10."""
    h, l, s = colorsys.rgb_to_hls(*base_rgb[:3])
    l = min(0.85, max(0.15, l + (frac - 0.5) * spread))
    return colorsys.hls_to_rgb(h, l, s)


def _exp_colors(exp_names):
    cmap_exp = plt.cm.tab10
    return {n: cmap_exp(i % 10) for i, n in enumerate(exp_names)}


def _exp_legend(fig, exp_color, exp_names, y_anchor=-0.02):
    handles = [plt.Line2D([0], [0], color=exp_color[n], lw=2, label=short_name(n))
               for n in exp_names]
    fig.legend(handles=handles, loc="lower center", ncol=min(len(exp_names), 4),
               fontsize=8, bbox_to_anchor=(0.5, y_anchor), frameon=True)


def _raw_conc(combined, well_row):
    conc = combined.get("concentration_raw")
    if conc is None:
        return None
    val = conc[well_row["row_idx"][0]]
    try:
        return float(val)
    except (TypeError, ValueError):
        return None


def plot_target_across_chips(combined, label_order=None, figsize_per_cell=(6, 4),
                             title=None, alpha_indiv=0.10, with_all_curves=True,
                             sharey=False, y_scale=None):
    """Copied from lofo_class_balance_analysis.ipynb §10 (unmodified)."""
    wells = build_well_table(combined)
    exp_names = sorted(wells["dataset_id"].unique())
    exp_color = _exp_colors(exp_names)

    labels_seen = list(dict.fromkeys(wells.sort_values("dataset_id")["label"]))
    if label_order is not None:
        all_labels = [l for l in label_order if l in labels_seen]
        all_labels += [l for l in labels_seen if l not in all_labels]
    else:
        nc    = [l for l in labels_seen if str(l).startswith("NC")]
        nonnc = [l for l in labels_seen if not str(l).startswith("NC")]
        all_labels = sorted(nonnc) + sorted(nc)

    t = np.asarray(combined["resampler"].t_grid, dtype=float)
    n = len(all_labels)
    fig, axes = plt.subplots(1, n, figsize=(figsize_per_cell[0] * n, figsize_per_cell[1]),
                             squeeze=False, sharey="row" if sharey else False)
    axes = axes[0]

    for ax, label in zip(axes, all_labels):
        label_wells = wells[wells["label"] == label].copy()
        label_wells["_conc"] = [_raw_conc(combined, w) for _, w in label_wells.iterrows()]
        concs_sorted = sorted({c for c in label_wells["_conc"] if c is not None})
        n_concs = len(concs_sorted)

        for _, w in label_wells.iterrows():
            idx = w["row_idx"]
            curves = combined["curves"][idx]
            mean = curves.mean(axis=0)
            conc = w["_conc"]
            frac = (concs_sorted.index(conc) / (n_concs - 1)) if (conc is not None and n_concs > 1) else 0.5
            color = _shade(exp_color[w["dataset_id"]], 1 - frac)

            if with_all_curves:
                step = max(1, len(curves) // 30)
                for c in curves[::step]:
                    ax.plot(t, c, color=color, lw=0.4, alpha=alpha_indiv, rasterized=True)
            ax.plot(t, mean, color=color, lw=1.5)

        ax.set_title(str(label), fontsize=11, fontweight="bold")
        ax.set_xlabel("Time", fontsize=8)
        ax.grid(True, linestyle="--", alpha=0.35)
        ax.tick_params(labelsize=7)
        if y_scale is not None:
            ax.set_ylim(*y_scale)

    axes[0].set_ylabel("Signal", fontsize=9)
    fig.suptitle(title or "Same target across chips  (colour=chip, shade=concentration)",
                fontsize=12, fontweight="bold")
    _exp_legend(fig, exp_color, exp_names)
    plt.tight_layout(rect=[0, 0.08, 1, 0.93])
    plt.show()


print("Visualisation helpers defined.")


### 4a. Baseline — current production alignment (no PC-TTP), for comparison

`cdt.combine_group` as it runs today (excludes PC, raw-acquisition zeroing). PC won't
appear here since it's excluded — this is just the "before" reference for §4b.

In [ ]:
combined_baseline = cdt.combine_group(exp_paths, GROUP_NAME, curve_type=CURVE_TYPE)
plot_target_across_chips(
    combined_baseline,
    title=f"BASELINE (current production, no PC-TTP alignment) | {GROUP_NAME} | {CURVE_TYPE}",
)


### 4b. PC-TTP-aligned — pick one (held-out chip, anchor method) to inspect

Change `HELD_OUT_CHIP`/`ANCHOR_METHOD` and re-run for a focused look. `HELD_OUT_CHIP`
is the chip treated as "unseen" for anchor computation (fold-scoped) -- it's still
included in the plot, using an anchor it had no part in computing, so you can see
directly whether the "unseen chip's TTP is earlier than the anchor" edge case (§3 of
the design doc) actually degrades gracefully or not.

In [ ]:
HELD_OUT_CHIP = folder_names[0]   # e.g. treat the first chip as "unseen"
ANCHOR_METHOD = "min"             # "min" | "percentile"
ANCHOR_PCT    = 10

result = combine_group_pc_aligned(exp_paths, CURVE_TYPE, held_out_chip=HELD_OUT_CHIP,
                                  anchor_method=ANCHOR_METHOD, anchor_pct=ANCHOR_PCT)
combined_aligned, pc_ttp_local, shifts, anchor = result

plot_target_across_chips(
    combined_aligned,
    title=(f"PC-TTP ALIGNED | held_out={short_name(HELD_OUT_CHIP)} | "
          f"anchor={ANCHOR_METHOD} | {GROUP_NAME} | {CURVE_TYPE}"),
)


### 4c. Full sweep — every chip held out, both anchor methods

8 figures (4 held-out chips × 2 anchor methods), each with one subplot per label —
large output, run deliberately. Compare against §4a's single baseline figure.

In [ ]:
for held_out in folder_names:
    for method in ["min", "percentile"]:
        print(f"\n=== held_out={short_name(held_out)}  anchor={method} ===")
        result = combine_group_pc_aligned(exp_paths, CURVE_TYPE, held_out_chip=held_out,
                                          anchor_method=method, anchor_pct=ANCHOR_PCT)
        if result is None:
            continue
        combined_i, _, _, _ = result
        plot_target_across_chips(
            combined_i,
            title=f"PC-TTP ALIGNED | held_out={short_name(held_out)} | anchor={method}",
        )


## 5. Visual check 2 — per-chip before/after (resampler-visualization style)

Mirrors §8's `plot_resampler_effect_for_exp` in `lofo_class_balance_analysis.ipynb`:
1 figure per chip, 1 subplot per well, 2 lines (before vs. after), x time-normalised
so both fit on [0, 1] regardless of the alignment step changing each chip's duration.
This shows *what the alignment mechanically did* to each chip -- complementary to §4's
"did it help" check.

In [ ]:
def plot_pc_alignment_effect_for_exp(exp_path, combined_before, combined_after,
                                     before_label="baseline", after_label="PC-TTP aligned"):
    mask_before = combined_before["dataset_id"] == exp_path.name
    mask_after  = combined_after["dataset_id"] == exp_path.name

    wells_before = build_well_table(combined_before)
    wells_before = wells_before[wells_before["dataset_id"] == exp_path.name]
    wells_after  = build_well_table(combined_after)
    wells_after  = wells_after[wells_after["dataset_id"] == exp_path.name]

    t_before = np.asarray(combined_before["resampler"].t_grid, dtype=float)
    t_after  = np.asarray(combined_after["resampler"].t_grid, dtype=float)
    x_before = (t_before - t_before[0]) / (t_before[-1] - t_before[0])
    x_after  = (t_after - t_after[0]) / (t_after[-1] - t_after[0])

    wells = sorted(set(wells_before["well_id"]) | set(wells_after["well_id"]))
    ncols = min(5, len(wells))
    nrows = int(np.ceil(len(wells) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 2.6 * nrows), squeeze=False)

    for i, well in enumerate(wells):
        ax = axes[i // ncols, i % ncols]
        wb = wells_before[wells_before["well_id"] == well]
        wa = wells_after[wells_after["well_id"] == well]
        label = wb.iloc[0]["label"] if len(wb) else wa.iloc[0]["label"]

        if len(wb):
            mean_b = combined_before["curves"][wb.iloc[0]["row_idx"]].mean(axis=0)
            ax.plot(x_before, mean_b, color="#2a78d6", lw=1.4, label=before_label)
        if len(wa):
            mean_a = combined_after["curves"][wa.iloc[0]["row_idx"]].mean(axis=0)
            ax.plot(x_after, mean_a, color="#eb6834", lw=1.4, ls="--", label=after_label)

        ax.set_title(f"well {well.split('::')[-1]} ({label})", fontsize=9)
        ax.set_xlabel("Normalised time", fontsize=7)
        ax.tick_params(labelsize=7)
        ax.legend(fontsize=6, loc="upper left")
        ax.grid(alpha=0.3)

    for j in range(len(wells), nrows * ncols):
        axes[j // ncols, j % ncols].axis("off")

    fig.suptitle(f"PC-TTP alignment effect — {exp_path.name}", fontsize=12, fontweight="bold")
    fig.tight_layout()
    plt.show()


for exp_path in exp_paths:
    plot_pc_alignment_effect_for_exp(exp_path, combined_baseline, combined_aligned)


## 6. Takeaways (fill in after inspecting the figures above)

- Raw PC TTP spread across chips (§2): _fill in_
- Does PC's own subplot in §4a/§4b visually converge across chips after alignment? _fill in_
- Does the held-out chip's PC curve (§4b, dashed-vs-not via colour) stay reasonably
  aligned, or does the clipping fallback (§3 of the design doc) produce a visibly
  worse result for it specifically? _fill in_
- `min` vs `percentile` anchor (§4c): does either look meaningfully better, or are
  they close enough not to matter with only 4 chips? _fill in_
- Do *other* targets (not just PC) look any more consistent across chips after
  alignment, or is the effect PC-specific? _fill in_

If PC converges well and the held-out-chip case degrades gracefully: worth pursuing as
a real `--pc_ttp_align`-style opt-in mode in `04_cross_dataset_training.py` (separate
implementation step, not done here). If not: the timestamp-misalignment hypothesis
probably isn't the (or isn't the whole) explanation for the domain-shift pattern in
`20260812-lofo_final_4_chip_clean_imbalance_vs_domain_shift.md`, and effort is better
spent on one of the other candidates from that doc's summary section.